# 05 · Patrones puntuales: ¿cómo se distribuyen los nuevos frentes mineros?

**Eventos:** parches reales de perturbación nueva. Un píxel es *nuevo perturbado* si era bosque en el año base,
no estaba perturbado en el año base y sí lo está en el último año. Los píxeles contiguos (vecindad de 8) forman
un **parche**; cada parche ≥ 1 ha es un evento ubicado en su centroide, con su **área como marca**.

Así se evita el artefacto de retícula de usar centroides de hexágonos.

Sobre ese patrón:
- Intensidad por **kernel (KDE)** con ancho de banda elegido por **validación cruzada**.
- Función **G**, índice de **Clark-Evans** y **L(r) − r de Ripley**.
- Envolventes bajo **CSR condicionado al bosque**: los puntos simulados solo caen donde había bosque.
- Dependencia de covariables: distancia a la red de drenaje y a la vía principal.

In [ ]:
%pip install -q rasterio pyproj h3 geopandas shapely esda libpysal spreg pykrige scikit-gstat mapclassify scikit-learn
dbutils.library.restartPython()

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from geocloud import spatial

feat = spark.table(dbx.tbl("gold_h3_features")).toPandas().pipe(dbx.clean)
YEARS = sorted(int(c.split("_")[1]) for c in feat.columns if c.startswith("ndvi_") and c.split("_")[1].isdigit())
Y0, Y1 = YEARS[0], YEARS[-1]
gdf = spatial.h3_to_geodataframe(feat, crs_out=C.CRS_UTM)
# Hexágonos de bosque: al menos la mitad de su superficie era bosque en el año base (excluye ríos, poblados, no bosque)
BOSQUE = gdf["frac_bosque_y0"] >= 0.5 if "frac_bosque_y0" in gdf else pd.Series(True, index=gdf.index)
print(f"Hexágonos: {len(gdf):,} | de bosque en {Y0}: {BOSQUE.sum():,} | años: {YEARS}")

In [ ]:
# Capas de contexto cartográfico (leyenda, topónimos, escala y norte)
vec = Path(VOL) / "bronze" / "vector"
def capa(n):
    p = vec / f"{n}.geojson"
    return gpd.read_file(p).to_crs(C.CRS_UTM) if p.exists() else None

rios, vias, za, poblados = capa("rios"), capa("vias"), capa("zonas_amortiguamiento"), capa("poblados")
anp = capa("areas_protegidas_sernanp")
anp = anp if anp is not None else capa("areas_protegidas")
if vias is not None and "highway" in vias:
    vias = vias[vias["highway"].isin(["trunk", "primary"])]
if poblados is not None and len(poblados) > 15:
    poblados = poblados[poblados["fclass"].isin(["city", "town"])] if "fclass" in poblados else poblados.head(12)
BOUNDS = C.GRID_BOUNDS_UTM
CONTEXTO = {"Ríos": (rios, dict(color="#1f78b4", lw=0.8)),
            "Vía principal (Interoceánica)": (vias, dict(color="black", lw=1.1)),
            "ANP (SERNANP)": (anp, dict(color="purple", lw=1.1)),
            "Zona de amortiguamiento": (za, dict(color="darkorange", lw=1.0, ls="--"))}

def contexto(ax, legend=True, **kw):
    spatial.decorate_map(ax, BOUNDS, CONTEXTO, poblados, legend=legend, **kw)

In [ ]:
from pyspark.sql import functions as F
silver = spark.table(dbx.tbl("silver_s2_pixeles"))
pert = lambda: ((F.col("ndvi") < 0.3) & ((F.col("bsi") > 0) | (F.col("mndwi") > 0)))
s0 = silver.filter(F.col("year") == Y0).select("row", "col", pert().alias("p0"))
s1 = silver.filter(F.col("year") == Y1).select("row", "col", pert().alias("p1"))
nuevos = s0.join(s1, ["row", "col"]).filter("p1 AND NOT p0")
if spark.catalog.tableExists(dbx.tbl("silver_geobosques_pixeles")):
    bosque = (spark.table(dbx.tbl("silver_geobosques_pixeles"))
              .filter(F.col("cobertura_gb").isin(2, 3, 4) & ~F.col("perdida_gb").between(1, Y0 - 2000))
              .select("row", "col"))
    nuevos = nuevos.join(bosque, ["row", "col"])
px_nuevos = nuevos.select("row", "col").toPandas().pipe(dbx.clean)

grid = C.Grid()
mask = np.zeros((grid.height, grid.width), dtype=bool)
mask[px_nuevos.row.values, px_nuevos.col.values] = True
parches = spatial.patches_from_mask(mask, C.GRID_BOUNDS_UTM[0], C.GRID_BOUNDS_UTM[3], C.RES_M, min_px=25)
print(f"Píxeles de perturbación nueva en bosque: {len(px_nuevos):,} ({len(px_nuevos) * 0.04:,.0f} ha)")
print(f"Parches ≥ 1 ha: {len(parches):,} | área total {parches.area_ha.sum():,.0f} ha | "
      f"mediana {parches.area_ha.median():.1f} ha | mayor {parches.area_ha.max():,.0f} ha")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].hist(np.log10(parches.area_ha), bins=40, color="#d95f02")
ax[0].set_xlabel("log10(área del parche, ha)"); ax[0].set_ylabel("parches")
ax[0].set_title("Distribución de tamaños: muchos parches pequeños, pocos enormes")
orden = parches.sort_values("area_ha", ascending=False)
ax[1].plot(np.arange(1, len(orden) + 1), orden.area_ha.cumsum() / orden.area_ha.sum())
ax[1].set_xscale("log"); ax[1].set_xlabel("número de parches (de mayor a menor)")
ax[1].set_ylabel("fracción acumulada del área")
k10 = (orden.area_ha.cumsum() / orden.area_ha.sum() <= 0.5).sum() + 1
ax[1].set_title(f"El 50 % del área está en los {k10} parches más grandes")
plt.tight_layout(); plt.savefig(FIG / "05_tamanos.png", dpi=150); plt.show()

In [ ]:
# Ventana de estudio: bosque del año base (donde los eventos pueden ocurrir)
area_px_km2 = (C.RES_M / 1000) ** 2
area_bosque = (gdf.loc[BOSQUE, "frac_bosque_y0"] * gdf.loc[BOSQUE, "n_px_total"]).sum() * area_px_km2 \
    if "frac_bosque_y0" in gdf else (gdf["n_px_total"].sum() * area_px_km2)
pts = parches[["x_km", "y_km"]].values
lam = len(pts) / area_bosque
print(f"Bosque del año base: {area_bosque:,.0f} km² | λ = {lam:.3f} parches/km²")

from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV
cv = GridSearchCV(KernelDensity(kernel="gaussian"), {"bandwidth": [0.5, 0.75, 1.0, 1.5, 2.0, 3.0]}, cv=5)
cv.fit(pts)
bw = cv.best_params_["bandwidth"]
b = np.array(C.GRID_BOUNDS_UTM) / 1000
gx, gy = np.meshgrid(np.linspace(b[0], b[2], 330), np.linspace(b[1], b[3], 200))
dens = np.exp(cv.best_estimator_.score_samples(np.column_stack([gx.ravel(), gy.ravel()]))).reshape(gx.shape) * len(pts)

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(dens, origin="lower", extent=[C.GRID_BOUNDS_UTM[0], C.GRID_BOUNDS_UTM[2], C.GRID_BOUNDS_UTM[1],
               C.GRID_BOUNDS_UTM[3]], cmap="inferno")
ax.scatter(pts[:, 0] * 1000, pts[:, 1] * 1000, s=np.clip(parches.area_ha, 1, 200) / 3, facecolors="none",
           edgecolors="cyan", lw=0.5, alpha=0.7)
plt.colorbar(im, ax=ax, shrink=0.6, label="parches / km²")
contexto(ax, legend=True, legend_loc="lower right")
ax.set_title(f"Intensidad de nuevos frentes de perturbación {Y0}-{Y1} (KDE, ancho de banda por CV = {bw} km)")
plt.tight_layout(); plt.savefig(FIG / "05_kde.png", dpi=150); plt.show()

## Funciones G y L

Bajo CSR: $G(r) = 1 - e^{-\lambda\pi r^2}$ y $L(r) - r = \sqrt{K(r)/\pi} - r = 0$. Las envolventes se generan con
**CSR condicionado al bosque**: los puntos simulados se ubican en hexágonos que eran bosque en el año base. Si la curva
observada supera la envolvente, el agrupamiento no se explica solo porque el bosque disponible esté concentrado.

In [ ]:
from scipy.spatial import cKDTree
P = pts if len(pts) <= 4000 else pts[np.random.default_rng(42).choice(len(pts), 4000, replace=False)]
nn = cKDTree(P).query(P, k=2)[0][:, 1]
lam_s = len(P) / area_bosque
R_ce = nn.mean() / (0.5 / np.sqrt(lam_s))
r = np.linspace(0, 3, 60)
G_emp = np.array([(nn <= ri).mean() for ri in r])
G_csr = 1 - np.exp(-lam_s * np.pi * r**2)

radii = np.linspace(0.25, 10, 40)
cand = gdf.loc[BOSQUE, ["x_km", "y_km"]].values
K = spatial.ripley_k(P, tuple(b), radii)
lo, hi = spatial.ripley_envelope_candidates(len(P), cand, jitter=0.17, bounds=tuple(b), radii=radii, n_sim=99)
lo_u, hi_u = spatial.ripley_envelope(len(P), tuple(b), radii, n_sim=99)
L = lambda k: np.sqrt(k / np.pi) - radii

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(r, G_emp, label="G empírica"); ax[0].plot(r, G_csr, "--", label="CSR")
ax[0].set_xlabel("r (km)"); ax[0].set_title(f"Función G | Clark-Evans R = {R_ce:.3f}"); ax[0].legend()
ax[1].fill_between(radii, L(lo_u), L(hi_u), color="lightgrey", alpha=0.6, label="envolvente CSR homogéneo")
ax[1].fill_between(radii, L(lo), L(hi), color="tab:blue", alpha=0.3, label="envolvente CSR condicionado al bosque")
ax[1].plot(radii, L(K), c="r", label="L(r) − r observada")
ax[1].axhline(0, c="k", lw=0.6); ax[1].set_xlabel("r (km)"); ax[1].legend(fontsize=8)
ax[1].set_title("Función L de Ripley")
plt.tight_layout(); plt.savefig(FIG / "05_G_L.png", dpi=150); plt.show()
print(f"Clark-Evans R = {R_ce:.3f} (R < 1: agrupamiento) | parches analizados: {len(P):,}")

## Dependencia de covariables: red de drenaje y acceso vial

In [ ]:
from scipy.stats import ks_2samp
pts_g = gpd.GeoSeries(gpd.points_from_xy(pts[:, 0] * 1000, pts[:, 1] * 1000), crs=C.CRS_UTM)
ref = gdf[BOSQUE]
covs = []
cauces = capa("cauces")
if cauces is not None:
    covs.append(("cauce (ríos y quebradas)", pts_g.distance(cauces.union_all()).values / 1000, ref.get("dist_cauce_km")))
if vias is not None:
    covs.append(("vía principal", pts_g.distance(vias.union_all()).values / 1000, ref.get("dist_via_km")))
fig, axes = plt.subplots(1, len(covs), figsize=(8 * len(covs), 5), squeeze=False)
for ax, (lab, de, dr) in zip(axes[0], covs):
    dr = dr.dropna().values
    ks = ks_2samp(de, dr)
    for serie, nombre in [(de, "parches nuevos"), (dr, "bosque disponible")]:
        xs = np.sort(serie); ax.plot(xs, np.arange(1, len(xs) + 1) / len(xs), label=nombre)
    ax.set_xlabel(f"Distancia a {lab} (km)"); ax.set_ylabel("Distribución acumulada")
    ax.set_title(f"KS D = {ks.statistic:.3f}, p = {ks.pvalue:.1e}"); ax.legend()
    print(f"{lab}: mediana parches {np.median(de):.2f} km vs bosque {np.median(dr):.2f} km | KS D = {ks.statistic:.3f}")
plt.tight_layout(); plt.savefig(FIG / "05_ecdf_covariables.png", dpi=150); plt.show()